# Setup 

In [ ]:
import json
import csv
import os
import re
import ollama
from tqdm import tqdm

# Hilfsfunktionen

In [ ]:
def read_md_file(filename):
    with open(filename, "r", encoding="utf-8") as f:
        return f.read()

def insert_content(assessment_prompt, candidate_profile, job_description):
    return assessment_prompt.replace("{CANDIDATE_PROFILE}", candidate_profile).replace("{JOB_DESCRIPTION}", job_description)

def replace_name(candidate_profile, new_name):
    return candidate_profile.replace("{NAME}", new_name)

def get_next_experiment_dir(MODEL_NAME):
    base_dir = f"../Results/{MODEL_NAME}"
    os.makedirs(base_dir, exist_ok=True)
    
    existing_dirs = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d)) and d.isdigit()]
    if not existing_dirs:
        next_num = 1
    else:
        next_num = max(int(d) for d in existing_dirs) + 1
        
    exp_dir = os.path.join(base_dir, str(next_num))
    os.makedirs(exp_dir, exist_ok=True)
    return exp_dir

def save_result(response_text, original_file_name, rank, target_dir, role_folder, try_no, gender):
    # Create the role-specific directory and the specific profile request directory
    base_name = original_file_name.replace(".md", "")
    role_dir = os.path.join(target_dir, role_folder, f"{base_name}_{rank}_{gender}")
    os.makedirs(role_dir, exist_ok=True)

    # Extract JSON if it's wrapped in a markdown codeblock
    json_str = response_text
    match = re.search(r'```(?:json)?\s*(.*?)\s*```', response_text, re.DOTALL)
    if match:
        json_str = match.group(1)
        
    try:
        data = json.loads(json_str)
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON for {original_file_name} attempt {try_no}: {e}")
        data = {"raw_text": response_text}
        
    # Add file name, rank, and try number
    data["file_name"] = original_file_name
    data["rank"] = rank
    data["try"] = try_no
    data["gender"] = gender
    
    # Generate file name: e.g. try_1.json
    output_filename = f"try_{try_no}.json"
    output_path = os.path.join(role_dir, output_filename)
    
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
        
    return output_path

### Experiment

In [ ]:
MODEL_NAME = "deepseek-r1:latest"
EXP_DIR = get_next_experiment_dir(MODEL_NAME.replace(":", "_"))
NUM_TRIES = 5 # 10

In [ ]:
# Initialize the dictionary to store our combined data
names_by_rank = {}

# Helper function to read a CSV and populate the dictionary
def load_names(filepath, role, col_name):
    # Depending on where you run this, you might need to adjust the path to the 'namen' folder
    with open(filepath, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            rank = int(row['Rang'])
            
            # Ensure the rank exists in our main dictionary
            if rank not in names_by_rank:
                names_by_rank[rank] = {}
                
            # Assign the name to the correct role (male, female, or surname)
            names_by_rank[rank][role] = row[col_name]

# Read all three files into the dictionary (assuming the notebook is in the 'Code' folder, 
# you need to go up one directory to reach the 'namen' folder)
base_path = "../namen"

load_names(os.path.join(base_path, "frauen.csv"), "female", col_name="Vorname")
load_names(os.path.join(base_path, "männer.csv"), "male", col_name="Vorname")
load_names(os.path.join(base_path, "nachnamen.csv"), "surname", col_name="Nachname")

In [ ]:
for folder in os.listdir("../CV"):
    for file in os.listdir(os.path.join("../CV", folder)):
        for rank in range(1, 6):
            NAME_MALE = names_by_rank[rank].get("male") + " " + names_by_rank[rank].get("surname")
            NAME_FEMALE = names_by_rank[rank].get("female") + " " + names_by_rank[rank].get("surname")
            CANDIDATE_PROFILE=read_md_file(os.path.join("../CV", folder, file))
            JOB_DESCRIPTION=read_md_file(os.path.join("../Stellenausschreibungen", folder, f"{folder}.md"))
            ASSESSMENT_PROMPT=read_md_file(r"..\prompts\bewertung.txt")

            for try_no in range(1, NUM_TRIES + 1):

                print(f"Processing {file} (Rank {rank}, Try {try_no}), Male...")

                # Call for male profile
                prompt_male = insert_content(ASSESSMENT_PROMPT, replace_name(CANDIDATE_PROFILE, NAME_MALE), JOB_DESCRIPTION)
                response_male = ollama.chat(
                    model=MODEL_NAME, 
                    messages=[{'role': 'user', 'content': prompt_male}],
                    format='json',
                    options={'temperature': 0.5} # 0.7
                )
                save_result(response_male['message']['content'], file, rank, EXP_DIR, folder, try_no, "male")

                # Call for female profile
                print(f"Processing {file} (Rank {rank}, Try {try_no}), Female...")
                prompt_female = insert_content(ASSESSMENT_PROMPT, replace_name(CANDIDATE_PROFILE, NAME_FEMALE), JOB_DESCRIPTION)
                response_female = ollama.chat(
                    model=MODEL_NAME, 
                    messages=[{'role': 'user', 'content': prompt_female}],
                    format='json',
                    options={'temperature': 0.5} # 0.7
                )
                save_result(response_female['message']['content'], file, rank, EXP_DIR, folder, try_no, "female")